# 08 — Tools & Agents: Hands-On Examples

> **Build progressively: from a single tool → tool calling → ReAct loop → full AgentExecutor → end-to-end research assistant.**

## What You'll Build
1. Define tools with `@tool` and Pydantic schemas
2. Bind tools to an LLM and see raw tool call responses
3. Implement the ReAct loop manually
4. Use `AgentExecutor` to automate the loop
5. Add conversation memory
6. Build a complete multi-tool research assistant

## Setup

In [ ]:
%pip install -qU langchain langchain-openai langchain-community tavily-python python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Verify keys are loaded
import os
print("OpenAI key:", "✅ set" if os.getenv("OPENAI_API_KEY") else "❌ missing")
print("Tavily key:", "✅ set" if os.getenv("TAVILY_API_KEY") else "❌ missing (optional for web search)")

---
## Part 1 — Tool Schemas

Three ways to define tools, from simplest to most controlled.

In [ ]:
from langchain_core.tools import tool

# Method 1: @tool decorator — reads type hints + docstring automatically
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.
    
    Args:
        city: The name of the city (e.g., 'London', 'Tokyo')
    """
    import random
    conditions = ["sunny", "cloudy", "rainy", "windy"]
    temp = random.randint(10, 30)
    return f"Weather in {city}: {temp}°C, {random.choice(conditions)}"

# Inspect the auto-generated schema
print("Name:", get_weather.name)
print("Description:", get_weather.description)
print("Args:", get_weather.args)
print("\nTest run:", get_weather.invoke({"city": "Paris"}))

In [ ]:
from pydantic import BaseModel, Field

# Method 2: Pydantic schema — precise field descriptions
class SearchInput(BaseModel):
    query: str = Field(description="The specific search query to look up")
    max_results: int = Field(default=3, description="Number of results to return (1-10)")

@tool("search_web", args_schema=SearchInput)
def search_web(query: str, max_results: int = 3) -> str:
    """Search the internet for current, real-time information and facts."""
    # Simulated results for this example
    return f"Search results for '{query}' (top {max_results}): [Article 1], [Article 2], [Article 3]"

print("Args schema:", search_web.args)
print("\nTest run:", search_web.invoke({"query": "langchain tools", "max_results": 2}))

In [ ]:
# All tools we'll use throughout this notebook
@tool
def calculator(expression: str) -> str:
    """Evaluate a Python mathematical expression. Use for arithmetic, percentages.
    
    Examples: '25 * 4', '100 / 7', '2 ** 10', '15 / 100 * 500'
    Args:
        expression: A valid Python math expression (no variables or imports)
    """
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

@tool
def get_current_date() -> str:
    """Get today's date and current time. Use when the user asks about dates or time."""
    from datetime import datetime
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

tools = [get_weather, search_web, calculator, get_current_date]
tools_map = {t.name: t for t in tools}
print(f"\n✅ {len(tools)} tools defined: {[t.name for t in tools]}")

---
## Part 2 — Tool Calling

Bind tools to an LLM and observe the raw tool call responses.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# Ask something that requires a tool
response = llm_with_tools.invoke([
    HumanMessage(content="What's the weather in Tokyo?")
])

print("Response type:", type(response).__name__)
print("Content (empty when using tools):", repr(response.content))
print("\nTool calls:")
for call in response.tool_calls:
    print(f"  name: {call['name']}")
    print(f"  args: {call['args']}")
    print(f"  id:   {call['id']}")

In [ ]:
# Ask something that does NOT need a tool — LLM answers directly
response_no_tool = llm_with_tools.invoke([
    HumanMessage(content="What is the capital of France?")
])

print("Tool calls:", response_no_tool.tool_calls)  # Empty list
print("Direct answer:", response_no_tool.content)

In [ ]:
# Parallel tool calling — LLM returns multiple calls at once
response_multi = llm_with_tools.invoke([
    HumanMessage(content="What's the weather in London and Tokyo, and what is 15% of 200?")
])

print(f"Number of parallel tool calls: {len(response_multi.tool_calls)}")
for call in response_multi.tool_calls:
    print(f"  → {call['name']}({call['args']})")

---
## Part 3 — The ReAct Loop (Manual)

Implement the Thought → Action → Observation cycle from scratch.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

def run_react_loop(question: str, max_steps: int = 10) -> str:
    """Manual ReAct loop: Thought → Action → Observation → repeat."""
    messages = [HumanMessage(content=question)]
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")
    
    for step in range(1, max_steps + 1):
        # Step 1: LLM reasons and decides
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        if not response.tool_calls:
            # LLM is done — no more tool calls
            print(f"\n✅ Final Answer (after {step-1} tool call(s)):")
            print(response.content)
            return response.content
        
        # Step 2: Execute each tool call
        print(f"\n[Step {step}] {len(response.tool_calls)} tool call(s):")
        for call in response.tool_calls:
            tool = tools_map[call["name"]]
            result = tool.invoke(call["args"])
            print(f"  → {call['name']}({call['args']})")
            print(f"  ← {result}")
            
            # Step 3: Feed result back to LLM
            messages.append(ToolMessage(
                content=result,
                tool_call_id=call["id"]
            ))
    
    return "Max steps reached"

# Test 1: Single tool
run_react_loop("What's the weather in Sydney?")

In [ ]:
# Test 2: Multi-step — needs multiple tool calls
run_react_loop("What is today's date, and what's 20% of 350?")

In [ ]:
# Test 3: No tool needed
run_react_loop("Explain what a neural network is in one sentence.")

---
## Part 4 — AgentExecutor

Let LangChain manage the loop automatically.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

# Prompt with scratchpad placeholder (required for AgentExecutor)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Think step by step before using tools."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),   # ← tool calls accumulate here
])

# Create agent (LLM + tools + prompt — no loop yet)
agent = create_tool_calling_agent(llm, tools, prompt)

# Wrap in AgentExecutor (adds the loop)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,           # Shows every step
    max_iterations=10,
    handle_parsing_errors=True,
)

print("✅ AgentExecutor ready!")

In [ ]:
# Run the agent — AgentExecutor handles everything automatically
result = agent_executor.invoke({
    "input": "What is the weather in Paris, and what is 18% of 450?"
})

print("\n" + "="*60)
print("FINAL ANSWER:")
print(result["output"])

In [ ]:
# Return intermediate steps for inspection
executor_with_steps = AgentExecutor(
    agent=agent, tools=tools,
    verbose=False,
    return_intermediate_steps=True,
)

result = executor_with_steps.invoke({"input": "What is today's date and weather in Berlin?"})

print("Intermediate steps:")
for i, (action, observation) in enumerate(result["intermediate_steps"], 1):
    print(f"  [{i}] Tool: {action.tool}")
    print(f"       Input: {action.tool_input}")
    print(f"       Output: {observation}")

print(f"\nFinal answer: {result['output']}")

---
## Part 5 — Agent with Memory

Add conversation persistence so the agent remembers previous turns.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Store per-session histories
session_store: dict = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# Wrap executor with history
agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

config = {"configurable": {"session_id": "demo-session"}}

# Turn 1
resp1 = agent_with_memory.invoke({"input": "My name is Alex and I like burritos."}, config=config)
print("Turn 1:", resp1["output"])

In [ ]:
# Turn 2 — agent remembers the name
resp2 = agent_with_memory.invoke({"input": "What is my name and what do I like?"}, config=config)
print("Turn 2:", resp2["output"])

In [ ]:
# Turn 3 — use a tool AND remember context
resp3 = agent_with_memory.invoke(
    {"input": "What's today's date, and based on my food preference, suggest a recipe."},
    config=config
)
print("Turn 3:", resp3["output"])

---
## Part 6 — Full Research Assistant

Put everything together: multiple tools + memory + a good system prompt.

In [ ]:
SYSTEM_PROMPT = """You are a smart research assistant.

You have access to tools for:
- Searching the web for current information
- Doing math calculations  
- Checking the weather in any city
- Getting today's date and time

Guidelines:
1. Think before acting — use the right tool for the job
2. For multi-part questions, handle each part systematically
3. Always give a clear, concise final answer
4. Remember what was discussed earlier in the conversation
"""

research_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

research_agent = create_tool_calling_agent(llm, tools, research_prompt)
research_executor = AgentExecutor(
    agent=research_agent,
    tools=tools,
    verbose=False,           # Set True to see steps
    max_iterations=10,
    handle_parsing_errors=True,
)

research_store: dict = {}
research_agent_with_memory = RunnableWithMessageHistory(
    research_executor,
    lambda sid: research_store.setdefault(sid, InMemoryChatMessageHistory()),
    input_messages_key="input",
    history_messages_key="chat_history",
)

def ask(question: str, session: str = "research") -> None:
    """Ask the research assistant a question."""
    print(f"\n🧑 You: {question}")
    config = {"configurable": {"session_id": session}}
    result = research_agent_with_memory.invoke({"input": question}, config=config)
    print(f"🤖 Assistant: {result['output']}")

print("✅ Research assistant ready!")

In [ ]:
# Test the complete assistant
ask("What is today's date?")

In [ ]:
ask("What's the weather in New York and London?")

In [ ]:
ask("If I invest $10,000 at 8% annual return for 5 years, how much will I have? Show the calculation.")

In [ ]:
ask("Based on the investment result you just calculated, what if I doubled my initial investment?")

---
## Summary

| Concept | What You Learned |
|---|---|
| **Tool schema** | `@tool` reads type hints + docstring to auto-generate JSON schema |
| **Tool binding** | `llm.bind_tools(tools)` — LLM sees schemas, not your code |
| **Tool call response** | `response.tool_calls` — list of `{name, args, id}` dicts |
| **ToolMessage** | Feed results back using `ToolMessage(content, tool_call_id)` |
| **ReAct loop** | Thought→Action→Observation, repeat until no tool calls |
| **AgentExecutor** | Automates the loop, handles errors, tracks iterations |
| **Memory** | `RunnableWithMessageHistory` adds per-session conversation history |

## Next Steps
- Replace simulated tools with real APIs (Tavily for search, OpenWeatherMap for weather)
- Try `tool_choice="required"` to force tool use
- Migrate to `create_react_agent` (LangGraph) for production use

## ➡️ Next Section
[09 — LangGraph Core Concepts](../09_langgraph_core_concepts/theory.md)